# Replicação passo a passo: classificação de tópicos jurídicos (entrega A1)

Neste caderno eu **executo de verdade**, célula a célula, tudo o que pode ser recomputado offline na entrega **A1, "consenso estável por área"**: normalização do texto, árvore e fusão de duplicados, extração de leis, o filtro dos sintéticos, os três leitores, o roteador, o OOF 5x5 completo, a calibração, a previsão de Jeffreys e, no fim, o `main()` de `scripts/gerar_publicacao.py` inteiro, do zero, conferido por md5. Não há reaproveitamento de resultados entre etapas (a etapa pesada roda duas vezes: uma para inspecionar, outra dentro do `main()`) e nenhuma função é substituída. As funções são as próprias de `scripts/`; nada é reimplementado aqui. Demora de propósito (dezenas de minutos em CPU).

**Regras do case que o pipeline respeita**
- Nenhum modelo de linguagem participa da decisão: só TF-IDF, ComplementNB, BM25, LSA, regressão logística e vetores já calculados.
- A decisão roda **offline** e nunca carrega nenhum modelo de embedding.
- A validação é aninhada e id-disjunta (5 repetições x 5 partes de teste sobre as 100 questões da folha); limiares e calibradores são aprendidos só no treino de cada parte.

**O que NÃO é re-executado aqui (e por quê)**
1. **Geração dos exemplos sintéticos** por Claude Sonnet: é um LLM, permitido só *antes* da decisão como produção de dados. A saída está versionada em `cache/sinteticos/` (a seção 6 mostra contagens e um exemplo; o prompt está em `prompts/gerar_sinteticos.md`). Não é determinística e custa API, então não faz sentido repeti-la.
2. **Metadados do Senado** (`scripts/atualizar_normas.py`): exige internet. Há uma célula protegida pela flag `RODAR_REDE = False` (seção 10); por padrão ela só avisa que foi pulada.
3. **Vetores (embeddings) Qwen3 e e5**: são artefatos de *preparação* (`scripts/embeddings.py`), permitidos antes da decisão (texto vira vetor). Por escolha, não são regenerados neste caderno: o pipeline os lê de `cache/embeddings/`, e a decisão nunca chama o modelo. Detalhes na seção 4.

O resultado final é conferido por **md5** (`publicacao.csv`) e pelo cabeçalho de `previsao.txt`.

## 1. Ambiente

Confiro a versão do Python, os pacotes contra `requirements.txt` e as threads de CPU. Também desligo a escrita de `__pycache__`, para a pasta terminar limpa.

In [1]:
import os, sys, io, json, re, time, shutil, hashlib, tempfile, contextlib, subprocess
import importlib.metadata as md_
from pathlib import Path
import numpy as np, pandas as pd

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
BASE = Path.cwd()
assert (BASE / "replicar.sh").exists(), "abra o caderno dentro da pasta final_otimo"
sys.path.insert(0, str(BASE / "scripts"))
pd.set_option("display.width", 200, "display.max_colwidth", 90)
T0 = time.time()
TEMPOS = {}
RODAR_REDE = False   # True: consulta a API do Senado na seção 10 (precisa de internet)
HF_HUB_OFFLINE = os.environ.setdefault("HF_HUB_OFFLINE", "1")

def curto(txt, n=1500):
    """Imprime no máximo n caracteres (mantém as saídas do caderno pequenas)."""
    txt = str(txt)
    print(txt if len(txt) <= n else txt[:n] + f"\n[... {len(txt) - n} caracteres omitidos]")

print("Python", sys.version.split()[0])
linhas = []
for l in (BASE / "requirements.txt").read_text().splitlines():
    if "==" in l and not l.startswith("#"):
        pkg, fix = l.split("==")
        inst = md_.version(pkg)
        linhas.append((pkg, fix, inst, "OK" if inst == fix else "DIFERENTE"))
print(pd.DataFrame(linhas, columns=["pacote", "fixado", "instalado", "status"]).to_string(index=False))
print("\nCPUs lógicas:", os.cpu_count(), "| threads BLAS (OMP_NUM_THREADS):", os.environ.get("OMP_NUM_THREADS", "padrão"), "| joblib n_jobs da validação: 8")

Python 3.14.9
         pacote fixado instalado status
          numpy  2.5.4     2.5.4     OK
         pandas  3.1.0     3.1.0     OK
          scipy 1.18.1    1.18.1     OK
   scikit-learn  1.9.1     1.9.1     OK
         joblib  1.6.0     1.6.0     OK
  threadpoolctl  3.7.0     3.7.0     OK
python-dateutil  4.0.0     4.0.0     OK
            six 1.17.0    1.17.0     OK
         tzdata 2026.6    2026.6     OK

CPUs lógicas: 12 | threads BLAS (OMP_NUM_THREADS): padrão | joblib n_jobs da validação: 8


## 2. Anexos do case e normalização do texto

Carrego as 659 questões, a árvore de temas e a folha de verdade (100 rotuladas; as outras 559 são as que precisam ser classificadas). Depois mostro, para a **Q0099** (HTML do Word) e a **Q0490** (letras de outro alfabeto no lugar das latinas), o texto antes e depois de `TextCleaner`, que remove HTML (inclusive comentários condicionais do Word e `<style>`), decodifica entidades e troca letras cirílicas/gregas por homóglifos latinos.

In [2]:
questoes = [json.loads(l) for l in (BASE / "questoes.jsonl").read_text(encoding="utf-8").splitlines()]
temas = pd.read_csv(BASE / "temas.csv")
folha = pd.read_csv(BASE / "folha_verdade.csv")
print(f"questões: {len(questoes)} | temas: {temas.shape} | folha: {folha.shape} (fora da taxonomia: {int(folha.fora_da_taxonomia.sum())})")
print("níveis da árvore:", temas.nivel.value_counts().sort_index().to_dict(), "| status:", temas.status.value_counts().to_dict())
q = questoes[0]
print("\nchaves da questão:", list(q))
print(q["id"], "|", q.get("disciplina_origem"), "|", q.get("assunto_origem"))
curto(re.sub(r"\s+", " ", q["enunciado_html"]), 500)

questões: 659 | temas: (377, 8) | folha: (100, 5) (fora da taxonomia: 8)
níveis da árvore: {0: 4, 1: 111, 2: 262} | status: {'active': 375, 'inactive': 2}

chaves da questão: ['id', 'enunciado_html', 'comando', 'alternativas', 'disciplina_origem', 'assunto_origem']
Q0001 | Processo Penal | Procedimento do Júri
No que se refere ao Tribunal do Júri, analise as assertivas abaixo.<br><br>I - O desaforamento é admitido por interesse da ordem pública, em razão de dúvida sobre a imparcialidade do júri, em razão de dúvida sobre a segurança pessoal do réu e não realização de julgamento, no período de um ano a contar da preclusão da pronúncia, em virtude de comprovado excesso de serviço.<br><br>II - A natureza jurídica da impronúncia é de uma decisão terminativa e está sujeita a Recurso em Sentido Estrito.<br><
[... 132 caracteres omitidos]


In [3]:
from preprocessor import TextCleaner
por_id = {r["id"]: r for r in questoes}

def nao_latinas(s):
    return [c for c in s if ord(c) > 127 and ("CYRILLIC" in __import__("unicodedata").name(c, "") or "GREEK" in __import__("unicodedata").name(c, ""))]

r = por_id["Q0099"]["enunciado_html"]
m_ = re.search(r"<!--\[if|<style|<o:p|<span|<p", r)
j = max(m_.start() - 20, 0) if m_ else 0
print(f"Q0099 | tamanho antes: {len(r)} caracteres; depois: {len(TextCleaner.clean(r))}")
print("Q0099 | antes  (trecho com marcação HTML):", repr(r[j:j + 160]))
print("Q0099 | depois (início):", repr(TextCleaner.clean(r)[:160]))
r = por_id["Q0490"]
bruto = " ".join([r.get("enunciado_html") or "", r.get("comando") or "", " ".join((r.get("alternativas") or {}).values())])
limpo = TextCleaner.extract_question_text(r.get("enunciado_html"), r.get("comando"), r.get("alternativas"))
print("\nQ0490 | letras não latinas antes:", len(nao_latinas(bruto)), "| depois:", len(nao_latinas(limpo)))
i = next((k for k, c in enumerate(bruto) if nao_latinas(c)), 0)
print("Q0490 | antes :", repr(bruto[max(0, i - 25): i + 45]))
print("Q0490 | depois:", repr(TextCleaner.clean(bruto[max(0, i - 25): i + 45])))

Q0099 | tamanho antes: 709 caracteres; depois: 691
Q0099 | antes  (trecho com marcação HTML): 'Filomena, camelô Irregular, é flagrada por dois agentes da fiscalização municipal, ocasião em que estes anunciam que irão lavrar um auto de infração e apreender'
Q0099 | depois (início): 'Filomena, camelô Irregular, é flagrada por dois agentes da fiscalização municipal, ocasião em que estes anunciam que irão lavrar um auto de infração e apreender'

Q0490 | letras não latinas antes: 2 | depois: 0
Q0490 | antes : 'ioeconômica do agente. prорõe a adoção de um conceito que limita a cul'
Q0490 | depois: 'ioeconômica do agente. propõe a adoção de um conceito que limita a cul'


## 3. Árvore de temas e fusão dos nós duplicados

Só os temas de **nível 1 ativos** são classes. `Contexto` (`scripts/dados.py`) reúne texto normalizado, taxonomia, BM25 e LSA, os vetores do cache e os sintéticos filtrados. Nele, `scripts/estrutura.py` funde os nós de nível 1 duplicados (sem filhos, sem descrição e com título quase igual ao de um irmão, mais os "(Parte N)") no irmão correspondente; um duplicado nunca é previsto. Criar o contexto leva cerca de um minuto.

In [4]:
import estrutura
from dados import Contexto
from validacao import N_REPS, N_FOLDS

ctx = Contexto("qwen")
tax, clf = ctx.tax, ctx.clf
print(f"raízes: {[tax.titulo[r] for r in tax.roots]}")
print(f"temas de nível 1 ativos: {len(tax.act_l1)} | duplicados fundidos: {len(ctx.dup)} | classes efetivas: {int(ctx.mask.sum())}")
print(f"limiar de Jaccard fixado a priori: {estrutura.LIMIAR_JACCARD}\n")
print(pd.DataFrame([(d, tax.titulo[d], a, tax.titulo[a]) for d, a in sorted(ctx.dup.items())],
                   columns=["id_dup", "duplicado", "id_alvo", "fundido em"]).to_string(index=False))

raízes: ['Criminologia', 'Direito Penal - Legislação Especial', 'Penal', 'Processo Penal']
temas de nível 1 ativos: 109 | duplicados fundidos: 12 | classes efetivas: 97
limiar de Jaccard fixado a priori: 0.6

 id_dup                                           duplicado  id_alvo                                fundido em
   6718 Dos Crimes Contra a Administração Pública (Parte 1)     5087 Dos Crimes Contra a Administração Pública
   6719 Dos Crimes contra a Administração Pública (Parte 2)     5087 Dos Crimes Contra a Administração Pública
   6720            Dos Crimes Contra o Patrimônio (Parte 1)     5157            Dos Crimes Contra o Patrimônio
   6721            Dos Crimes contra o Patrimônio (Parte 2)     5157            Dos Crimes Contra o Patrimônio
   6726                            Teoria da Pena (Parte 1)     5199                            Teoria da Pena
   6727                            Teoria da Pena (Parte 2)     5199                            Teoria da Pena
   6732       

## 4. Vetores, sintéticos e o filtro automático

**Vetores.** `cache/embeddings/qwen/` (Qwen3-Embedding-0.6B: questões, pseudo-documentos dos temas, sintéticos e descrições) e `cache/embeddings/e5/` (multilingual-e5-base, usados só pelo filtro) foram produzidos por `scripts/embeddings.py`. Isso é **preparação** (pré-processamento: texto vira vetor, com modelo local e sem LLM generativo), permitida antes da decisão. Por escolha, neste caderno **não os regenero** (nem Qwen nem e5): os leio do cache, exatamente como o pipeline faz. A decisão em si nunca chama o modelo de embedding. O manifesto abaixo registra modelo, revisão e SHA-256 de cada vetor.

**Sintéticos.** São 12 exemplos + 1 descrição por tema-alvo, gerados por Claude Sonnet com o prompt de `prompts/gerar_sinteticos.md` e versionados em `cache/sinteticos/`. Mostro contagens e um exemplo.

In [5]:
for sl in ("qwen", "e5"):
    mf = json.loads((BASE / "cache" / "embeddings" / sl / "manifest.json").read_text())
    print(sl, "|", mf["modelo"], "| revisão", str(mf["revisao"])[:10], "| gerado em", mf["data"])
    print("   ", {k: (v["n"], v["dim"], v["sha256"][:10]) for k, v in mf["arquivos"].items()})
arqs = sorted((BASE / "cache" / "sinteticos").glob("tema_*.jsonl"))
regs = [[json.loads(l) for l in a.read_text(encoding="utf-8").splitlines() if l.strip()] for a in arqs]
print(f"\ntemas-alvo com sintéticos: {len(arqs)} | exemplos: {sum(sum(r.get('tipo')=='exemplo' for r in L) for L in regs)} | descrições: {sum(sum(r.get('tipo')=='descricao' for r in L) for L in regs)}")
ex1 = next(r for r in regs[0] if r.get("tipo") == "exemplo")
print("exemplo (", arqs[0].name, "):")
curto(json.dumps(ex1, ensure_ascii=False, indent=1), 900)
print("\nprompt de geração (início de prompts/gerar_sinteticos.md):")
curto((BASE / "prompts" / "gerar_sinteticos.md").read_text(encoding="utf-8"), 700)

qwen | Qwen/Qwen3-Embedding-0.6B | revisão 97b0c614be | gerado em 2026-11-03
    {'questoes': (659, 1024, '1b8d804171'), 'pseudo': (109, 1024, '9b68c35b3c'), 'sint': (1212, 1024, 'd9e6f5c477'), 'sint_desc': (101, 1024, 'f40221fdbd')}
e5 | intfloat/multilingual-e5-base | revisão d128750597 | gerado em 2026-11-03
    {'pseudo': (109, 768, 'a053377f37'), 'sint': (1212, 768, '827e044c94'), 'sint_desc': (101, 768, 'b861b0b6b6')}

temas-alvo com sintéticos: 101 | exemplos: 1212 | descrições: 101
exemplo ( tema_4366.jsonl ):
{
 "tema_id": 4366,
 "tipo": "exemplo",
 "estilo": "caso_concreto",
 "enunciado": "Um homem foi vítima de um assalto, tendo sido surpreendido de madrugada em sua casa por criminosos armados, sem qualquer contribuição de sua parte para o crime. Na classificação de Mendelsohn, trata-se de vítima",
 "comando": "Assinale a alternativa correta.",
 "alternativas": {
  "A": "inocente ou ideal.",
  "B": "provocadora.",
  "C": "voluntária.",
  "D": "simuladora.",
  "E": "por ignor

### Filtro automático dos sintéticos (executado)

`scripts/filtrar_sinteticos.py` faz o *round-trip*: uma regressão logística sobre os vetores e5, treinada nos outros exemplos (6 partes, estratificadas por tema) mais descrições e pseudo-documentos, tem de colocar o tema do exemplo no top-2; depois remove quase-duplicatas (cosseno >= 0,98). Rodo o script de verdade gravando em pasta temporária (assim a amostra de revisão humana de `cache/` nunca é sobrescrita) e comparo o `sinteticos_filtrados.jsonl` e o relatório com os do cache. Os vetores e5 vêm do cache, como no pipeline (a regeneração dos vetores fica fora do caderno, ver acima).

In [6]:
tmp_f = Path(tempfile.mkdtemp())
t = time.time()
r = subprocess.run([sys.executable, "-B", "scripts/filtrar_sinteticos.py", "--saida", str(tmp_f)], check=True, capture_output=True, text=True)
TEMPOS["filtro"] = time.time() - t
curto(r.stdout, 600)
for nome in ("sinteticos_filtrados.jsonl", "sinteticos_filtro_relatorio.json"):
    igual = (tmp_f / nome).read_bytes() == (BASE / "cache" / nome).read_bytes()
    print(f"{nome}: idêntico ao cache (byte a byte) -> {'OK' if igual else 'DIFERENTE'}")
print(f"({TEMPOS['filtro']:.0f}s; amostra de revisão gerada só na pasta temporária; a revisada de cache/ ficou intacta: {(BASE/'cache'/'sinteticos_amostra_revisao.csv').exists()})")

Filtro sintetico: 1212 exemplos; top-1 correto 52.2%; tema no top-2 73.0% (327 removidos); quase-duplicatas cos>=0.98: 9 removidas; mantidos 876/1212 = 72.3%

sinteticos_filtrados.jsonl: idêntico ao cache (byte a byte) -> OK
sinteticos_filtro_relatorio.json: idêntico ao cache (byte a byte) -> OK
(27s; amostra de revisão gerada só na pasta temporária; a revisada de cache/ ficou intacta: True)


## 5. Leis citadas

`scripts/extrair_leis.py` (passo 2 do `replicar.sh`) acha por regex as normas e os artigos citados em cada questão. Rodo o script agora, gravando em arquivo temporário, e comparo com o `cache/leis_por_questao.csv` usado no restante. Quando a questão cita uma única norma que aponta um único tema de nível 1, esse tema é forçado na decisão (e dá um bônus de 2,8 no logit do score).

In [7]:
tmp_leis = Path(tempfile.mkdtemp()) / "leis.csv"
subprocess.run([sys.executable, "-B", "scripts/extrair_leis.py", "--saida", str(tmp_leis)], check=True, capture_output=True)
a, b = pd.read_csv(tmp_leis).fillna(""), pd.read_csv(BASE / "cache" / "leis_por_questao.csv").fillna("")
print("extrair_leis reproduz o cache:", "OK" if a.astype(str).equals(b.astype(str)) else "DIFERENTE")
leis = clf.leis_df
print(f"questões com norma citada: {int(leis.tem_lei.sum())} de {len(leis)}; com tema de nível 1 unívoco: {int((leis.tema_lei_n1.notna() & ~leis.tema_lei_n1.fillna('').str.contains(';')).sum())}")
ex_lei = leis[leis.tem_lei & leis.tema_lei_n1.notna()].iloc[0]
tid = int(str(ex_lei.tema_lei_n1).split(";")[0])
print("\nexemplo:", ex_lei.name, "| leis:", ex_lei.leis_citadas, "| artigos:", ex_lei.artigos_citados, "->", tax.caminho[tid])

extrair_leis reproduz o cache: OK
questões com norma citada: 220 de 659; com tema de nível 1 unívoco: 89

exemplo: Q0012 | leis: LEI-9099-1995 | artigos: 77 -> Direito Penal - Legislação Especial > Juizado Especial Criminal


## 6. Os três leitores e o consenso

Cada questão passa por três leitores independentes (`scripts/models.py`): **NB hierárquico** (raiz e depois tema, TF-IDF de palavras e caracteres e leis citadas; supervisionado, treinado nas 92 questões rotuladas), **BM25** sobre um documento por tema e **LSA** (TF-IDF + SVD de 50 dimensões). Mostro o tema que cada um sugere para quatro questões fora da folha. Quando os três coincidem há **consenso**, o primeiro requisito para publicar em Penal e Processo Penal (`scripts/decisao.py`).

In [8]:
lab = ctx.sup[ctx.sup.y != -1]
estado = clf.fit_hierarchy(lab)
fora_folha = [i for i in ctx.q.id if i not in set(ctx.sup.id)]
amostra = fora_folha[:3] + ["Q0400"]
idx = [ctx.qidx[i] for i in amostra]
P_nb = clf.predict_hierarchy(estado, ctx.q.iloc[idx])
nome = lambda col: tax.titulo[tax.act_l1[int(col)]]
rows = []
for j, (qid, i) in enumerate(zip(amostra, idx)):
    t = [int(P_nb[j].argmax()), int(ctx.S_bm[i].argmax()), int(ctx.S_ls[i].argmax())]
    rows.append((qid, nome(t[0]), nome(t[1]), nome(t[2]), "sim" if len(set(t)) == 1 else "não"))
print(pd.DataFrame(rows, columns=["questão", "NB", "BM25", "LSA", "consenso"]).to_string(index=False))

questão                               NB                             BM25                                 LSA consenso
  Q0001 Procedimento do Tribunal do Júri Procedimento do Tribunal do Júri    Procedimento do Tribunal do Júri      sim
  Q0004                      Competência                      Competência                         Competência      sim
  Q0006   Dos Crimes Contra o Patrimônio   Dos Crimes Contra o Patrimônio      Dos Crimes Contra o Patrimônio      sim
  Q0400   Teoria do Crime: Culpabilidade                       Prescrição Das Questões e Processos Incidentes      não


## 7. Roteador por área e o OOF 5x5 completo

O **roteador por área** (`scripts/roteador_area.py`) é uma regressão logística sobre o vetor Qwen3 do texto (sem disciplina nem assunto) que decide a raiz (Penal, Processo Penal, Legislação Especial ou Criminologia). Isso importa porque as questões novas (Q0360+) nunca trazem a disciplina de Legislação Especial. É treinado só com sintéticos filtrados, descrições e a folha do treino de cada parte.

`validacao.executar` roda, para cada uma das 25 partes de teste e para a produção (folha inteira, 659 questões), a regra S4 (`decisao.decidir_fold`) e a das áreas novas (`roteador_area.decidir_fold`), combinadas por raiz roteada, em paralelo. É a etapa pesada, executada aqui integralmente e cronometrada.

In [9]:
import roteador_area as ra, validacao as val
print("vetores Qwen3 das questões:", ctx.E_q.shape, "| sintéticos mantidos:", int(ctx.keep.sum()), "de", len(ctx.keep))
t = time.time()
ex, MOD_A = val.executar(ctx)
h = n = 0
for key, d in ex.items():
    if key == "prod":
        continue
    for i, rt in zip(d.id, d.rt):
        if ctx.ylab[i] >= 0:
            n += 1; h += int(tax.pai[tax.act_l1[ctx.ylab[i]]] == rt)
TEMPOS["oof"] = time.time() - t
print(f"roteador de raiz, acurácia OOF: {h}/{n} = {h/n:.3f}   ({time.time()-t:.0f}s)")

vetores Qwen3 das questões: (659, 1024) | sintéticos mantidos: 876 de 1212


roteador de raiz, acurácia OOF: 376/460 = 0.817   (161s)


## 8. Regra de decisão por área, funil e validação cruzada

A regra de publicação (`validacao.publicar`) tem três filtros em sequência: (1) **consenso** entre os leitores (Penal e Processo: NB = BM25 = LSA; Legislação Especial e Criminologia: BM25 = qwen-LR na raiz roteada, com a norma citada concordando quando houver); (2) **estabilidade**: o consenso se repete em pelo menos 80% de 25 subamostras de 80% do treino (`decisao.sacos_necessarios()` = 20 sacos); (3) **piso de score**: descarta o quartil inferior do score do consenso (corte por CV interna, por área). A tabela mostra o funil nas 559 de produção; depois vêm a precisão por repetição e por área no OOF 5x5 e os erros distintos (a **Q0043** tem rótulo `fora da taxonomia`, mas o texto trata de dosimetria da pena e o sistema a publica em Penal).

In [10]:
d = ex["prod"].copy()
pub_prod = val.publicar(d)
d = d[~d.id.isin(set(ctx.sup.id))]                       # as 559 fora da folha
c1 = d.r0.values
c2_ = c1 & (d.cnt.values >= val.decisao.sacos_necessarios())
c3_ = c2_ & (d.score.values >= d.qg.values)
assert (c3_ == pub_prod[~ex["prod"].id.isin(set(ctx.sup.id)).values]).all()
tab = pd.DataFrame(dict(area=[tax.titulo[r] for r in d.root], consenso=c1, estavel=c2_, publicadas=c3_)).groupby("area").agg(
    questoes=("consenso", "size"), com_consenso=("consenso", "sum"), e_estaveis=("estavel", "sum"), e_acima_do_piso=("publicadas", "sum"))
tab.loc["TOTAL"] = tab.sum()
print(tab.to_string())

                                     questoes  com_consenso  e_estaveis  e_acima_do_piso
area                                                                                    
Criminologia                               64            15          14                4
Direito Penal - Legislação Especial       122            42          39               22
Penal                                     205            86          78               52
Processo Penal                            168            56          51               25
TOTAL                                     559           199         182              103


In [11]:
av = val.avaliar(ex)
H = av["res"]
print(pd.DataFrame([(i + 1, a, b, f"{a/b:.3f}" if b else "-") for i, (a, b, _) in enumerate(av["por_rep"])],
                   columns=["repetição", "acertos", "publicadas", "precisão"]).to_string(index=False))
print(f"\nagregado: {H['K']}/{H['N']} = {H['prec']:.3f} | pior repetição {H['pior']:.3f} | cobertura OOF {H['cob']:.1%}")
print("\npor área (OOF acertos/publicadas de total; produção publicadas/total nas 559):")
print(pd.DataFrame([(tax.titulo[k], f"{v['oof_h']}/{v['oof_p']} de {v['oof_t']}", f"{v['prod_p']}/{v['prod_t']}") for k, v in val.por_area(ctx, av).items()],
                   columns=["área", "OOF", "produção"]).to_string(index=False))
oof = av["oof"]
wrong = oof[oof.pub & (oof.ramo != 1)].groupby("id").size().to_dict()
print("\nerros distintos (questão: repetições em que errou):", wrong or "nenhum")

 repetição  acertos  publicadas precisão
         1       22          23    0.957
         2       22          23    0.957
         3       22          23    0.957
         4       22          23    0.957
         5       25          26    0.962

agregado: 113/118 = 0.958 | pior repetição 0.957 | cobertura OOF 23.6%

por área (OOF acertos/publicadas de total; produção publicadas/total nas 559):
                               área          OOF produção
                       Criminologia  26/26 de 70     4/64
Direito Penal - Legislação Especial  20/20 de 96   22/122
                              Penal 46/51 de 203   52/205
                     Processo Penal 21/21 de 131   25/168

erros distintos (questão: repetições em que errou): {'Q0043': 5}


## 9. Calibração (Brier) e previsão de Jeffreys

A coluna `confianca` é uma probabilidade calibrada (regressão logística + isotônica, `scripts/calibracao.py`) de o ramo sugerido estar correto, ajustada em validação aninhada id-disjunta; mostro o escore de Brier e a confiança média contra a precisão observada. A previsão (`scripts/previsao.py`) usa o intervalo de Jeffreys sobre o OOF da própria entrega: k e n são acertos e publicadas médios por repetição; a cobertura prevista é a fração realmente publicada nas 559. O limite conservador de 0,94 (seleção aninhada entre ~35 candidatas, A1 escolhida em 25/25 dobras) foi calculado no desenvolvimento (ver `docs/ANEXO_EXPERIMENTOS.md`) e não é recalculado aqui.

In [12]:
import calibracao as cb, previsao
t = time.time()
oof_cal, cal_final = cb.oof_aninhado(ex, val.publicar, N_REPS, N_FOLDS)
TEMPOS['calibracao'] = time.time() - t
rel = cb.relatorio(oof_cal)
print(f"Brier da confiança calibrada: {rel['brier_new']:.3f} (todas as linhas) | só publicadas: {rel['brier_new_pub']:.3f}")
print(f"confiança calibrada média nas publicadas {rel['mean_pub']:.3f} x precisão observada {rel['prec_pub']:.3f} (n={rel['n_pub']})")
cc = val.cobertura_559(ctx, av["prod"])
c = previsao.cabecalho(H, cc)
print(f"\nJeffreys sobre o OOF da entrega: precisão prevista {c['prec']:.2f} (intervalo {c['prec_lo']:.2f}-{c['prec_hi']:.2f})")
print(f"cobertura prevista {c['cob']:.2f} = {cc['p559']}/{cc['n559']} (intervalo {c['cob_lo']:.2f}-{c['cob_hi']:.2f})")

Brier da confiança calibrada: 0.185 (todas as linhas) | só publicadas: 0.044
confiança calibrada média nas publicadas 0.964 x precisão observada 0.958 (n=118)

Jeffreys sobre o OOF da entrega: precisão prevista 0.94 (intervalo 0.82-1.00)
cobertura prevista 0.18 = 103/559 (intervalo 0.16-0.33)


## 10. Metadados do Senado (opcional, precisa de internet)

Se `RODAR_REDE = True` (definida na seção 1), rodo `scripts/atualizar_normas.py` e comparo o resultado com `cache/normas_senado.json`. Como o script grava em `cache/`, faço isso numa cópia temporária da pasta `cache/` e `scripts/` para não tocar a entrega. Por padrão a célula só avisa que foi pulada.

In [13]:
if not RODAR_REDE:
    print("PULADO: RODAR_REDE = False (a consulta ao Senado precisa de internet; cache/normas_senado.json é lido como está).")
else:
    espelho = Path(tempfile.mkdtemp()) / "repo"
    espelho.mkdir()
    shutil.copytree(BASE / "scripts", espelho / "scripts", ignore=shutil.ignore_patterns("__pycache__"))
    (espelho / "cache").mkdir()
    for a in ("lei_tema.csv", "leis_por_questao.csv", "normas_senado.json"):
        shutil.copy(BASE / "cache" / a, espelho / "cache" / a)
    shutil.copy(BASE / "temas.csv", espelho / "temas.csv")
    subprocess.run([sys.executable, "-B", "scripts/atualizar_normas.py", "--forcar"], cwd=espelho, check=True)
    novo = json.loads((espelho / "cache" / "normas_senado.json").read_text(encoding="utf-8"))["normas"]
    velho = json.loads((BASE / "cache" / "normas_senado.json").read_text(encoding="utf-8"))["normas"]
    dif = [k for k in set(novo) | set(velho) if {x: v for x, v in novo.get(k, {}).items() if x != "consultado_em"} != {x: v for x, v in velho.get(k, {}).items() if x != "consultado_em"}]
    print(f"normas: {len(novo)} novas x {len(velho)} no cache | diferentes: {len(dif)} ->", "OK" if not dif else "DIFERENTE", dif[:10])

PULADO: RODAR_REDE = False (a consulta ao Senado precisa de internet; cache/normas_senado.json é lido como está).


## 11. Produção do zero e conferência por md5

Por fim, o `main` de `scripts/gerar_publicacao.py` (o mesmo que o `./replicar.sh` chama) roda **inteiro, do zero**: recarrega os dados, refaz o OOF 5x5 e a produção, a calibração e grava `publicacao.csv`, `previsao.txt` e o modelo da API. Nada do que calculei acima é reaproveitado e nenhuma função é substituída. A saída vai para pasta temporária e **só substitui os arquivos da entrega se o md5 de `publicacao.csv` e o cabeçalho de `previsao.txt` forem idênticos**.

In [14]:
import gerar_publicacao as gen
saida = Path(tempfile.mkdtemp())
sys.argv = ["gerar_publicacao.py", "--saida", str(saida / "publicacao.csv"), "--previsao", str(saida / "previsao.txt"), "--modelo", str(saida / "modelo.pkl")]
buf = io.StringIO()
t = time.time()
with contextlib.redirect_stdout(buf):
    gen.main()
TEMPOS["main"] = time.time() - t
print(f"main (do zero) concluído em {time.time()-t:.0f}s; saída do script ({len(buf.getvalue().splitlines())} linhas), trechos principais:")
for l in buf.getvalue().splitlines():
    if l.strip().startswith(("OOF:", "Arquivos:")):
        curto(l, 260)

ESPERADO_MD5 = "0d9329da890c4816850747cdf3855f99"
CABECALHO = ["precisao_prevista: 0.94", "precisao_intervalo: 0.82-1.00", "cobertura_prevista: 0.18", "cobertura_intervalo: 0.16-0.33"]
md5 = lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest()
ok_pub = md5(saida / "publicacao.csv") == ESPERADO_MD5
ok_prev = (saida / "previsao.txt").read_text(encoding="utf-8").splitlines()[:4] == CABECALHO
print(f"\npublicacao.csv: {md5(saida / 'publicacao.csv')}  esperado {ESPERADO_MD5}  -> {'OK' if ok_pub else 'DIFERENTE'}")
print(f"previsao.txt (4 linhas do cabeçalho): {'OK' if ok_prev else 'DIFERENTE'}")
if ok_pub and ok_prev:
    for a in ("publicacao.csv", "previsao.txt"):
        shutil.copy(saida / a, BASE / a)
    print("\nRESULTADO: OK (md5 de publicacao.csv e cabeçalho de previsao.txt idênticos aos da entrega; arquivos gravados na pasta)")
else:
    print("\nRESULTADO: DIFERENTE (arquivos mantidos só em", saida, ")")

main (do zero) concluído em 170s; saída do script (7 linhas), trechos principais:
      OOF: 113/118 = 0.958 (pior repeticao 0.957); cobertura OOF 0.236; publicadas nas 559: 103/559
Arquivos: /tmp/tmphv2w4wlt/publicacao.csv (659 linhas), /tmp/tmphv2w4wlt/previsao.txt

publicacao.csv: 0d9329da890c4816850747cdf3855f99  esperado 0d9329da890c4816850747cdf3855f99  -> OK
previsao.txt (4 linhas do cabeçalho): OK

RESULTADO: OK (md5 de publicacao.csv e cabeçalho de previsao.txt idênticos aos da entrega; arquivos gravados na pasta)


### Resultado final

Decisões nas 659 questões, publicações nas 559 por área (a raiz sai de `temas.csv` pelo `tema_central_id`), o início do `previsao.txt` e alguns exemplos publicados.

In [15]:
pub = pd.read_csv(saida / "publicacao.csv")
caminho = temas.set_index("tema_id").caminho.to_dict()
pub["area"] = pub.tema_central_id.map(lambda t: tax.titulo[tax.pai[int(t)]])
pub["tema"] = pub.tema_central_id.map(caminho)
print("decisões nas 659:", pub.decisao.value_counts().to_dict())
nas559 = pub[~pub.id.isin(set(folha.id))]
print("\npublicações nas 559, por área:")
print(nas559.groupby("area").decisao.value_counts().unstack(fill_value=0).assign(total=lambda x: x.sum(axis=1)).to_string())
print("\nprevisao.txt (início):")
print("\n".join((saida / "previsao.txt").read_text(encoding="utf-8").splitlines()[:5]))
print("\nexemplos publicados:")
print(nas559[nas559.decisao == "publicar"].groupby("area").head(2)[["id", "tema", "confianca"]].to_string(index=False))
print("\ntempos (s):", {k: round(v) for k, v in TEMPOS.items()})
print(f"tempo total do caderno até aqui: {(time.time()-T0)/60:.1f} min")

decisões nas 659: {'revisar': 517, 'publicar': 142}

publicações nas 559, por área:
decisao                              publicar  revisar  total
area                                                         
Criminologia                                4       60     64
Direito Penal - Legislação Especial        22      100    122
Penal                                      52      153    205
Processo Penal                             25      143    168

previsao.txt (início):
precisao_prevista: 0.94
precisao_intervalo: 0.82-1.00
cobertura_prevista: 0.18
cobertura_intervalo: 0.16-0.33


exemplos publicados:
   id                                                              tema  confianca
Q0001                 Processo Penal > Procedimento do Tribunal do Júri      0.990
Q0004                                      Processo Penal > Competência      0.939
Q0022                 Penal > Dos Crimes Contra a Administração Pública      0.990
Q0025                 Penal > Dos Crimes Contra a Admin

## 12. Fechamento

- Método, justificativas e limitações: `relatorio/relatorio_tecnico.pdf`.
- Resultados de todos os experimentos, incluindo os descartados: `docs/ANEXO_EXPERIMENTOS.md`.
- `./replicar.sh` faz as etapas 5 (extração de leis) e 11 (pipeline completo) em um comando.

A precisão e a cobertura previstas em `previsao.txt` são estimativas com incerteza grande (n pequeno de questões distintas), e a precisão nas questões novas (Q0360+) não é observável com a folha disponível.